# Personal Project: Speech Analyser

## Introduction

As I round out my course on machine learning, I found that in some ways, I was losing motivation following a course, which has, undoubtedly, given me the blessing of learning code and machine learning, though through its structure, has meant that my curiosity is not piqued often in the way pursuing my other interests would. 

I have decided to begin a personal project that uses these skills, demands further skills and learning, and inspires me to keep learning while honing my other interests

I have always been passionate about languages, and as such the idea of a program that analyses speech and helps users refine their accent seemed, at first like a good idea. Upon reflection, this idea poses a few issues, namely around the assumption that there was one accent that speakers should aim for, which of course is not true. 

To further develop this idea, I would like to redirect the goal of this project, to still analyse speech and provide coaching, but more for users who would like to improve their public speaking or presentation skills with in-time feedback through analysis of speed, tone, volume and intonation. 

### Project Goals

xx

### Project Architecture

In [5]:
'''
[ Mic/File Upload ]
        ↓
[ Whisper STT ]
        ↓
[ Text + Timestamp Output ]
        ↓
[ Analyse: fillers, pitch, pauses, pace ]
        ↓
[ Score + Feedback ]
        ↓
[ Simple UI Output (text + visuals) ]
'''

'\n[ Mic/File Upload ]\n        ↓\n[ Whisper STT ]\n        ↓\n[ Text + Timestamp Output ]\n        ↓\n[ Analyse: fillers, pitch, pauses, pace ]\n        ↓\n[ Score + Feedback ]\n        ↓\n[ Simple UI Output (text + visuals) ]\n'

## Audio Input and Acoustic Analysis

I will build the project by using audio files recorded by myself, before testing on other speech libraries and ultimately opening up for other users.

Using wave and librosa, the audio file is loaded and resampled if necessary. 16,000Hz mono is the ideal format for audio analysis. Then, initial acoustic analysis is performed to assess vocal energy (pitch and intensity) and voice stability (shimmer and jitter).

In [ ]:
import os
import wave
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import whisper

# Load and inspect the audio file
def load_audio(filename, target_sr=16000):
    # Load the audio file
    with wave.open(filename, 'rb') as wav_file:
        # Get audio file properties
        channels = wav_file.getnchannels()
        rate = wav_file.getframerate()
        frames = wav_file.readframes(wav_file.getnframes())
        # Convert byte data to numpy array
        # Assuming the audio is 16000Hz mono
        audio = np.frombuffer(frames, dtype=np.int16)
        duration = len(audio) / rate
        print(f"Name: '{filename}', Channels: {channels}, Sample Rate: {rate}, Duration: {duration:.2f} seconds.")

    # Check if resampling to 16000Hz is necessary
    if rate != target_sr:
        # Load the audio file
        audio, rate = librosa.load(filename, sr=target_sr, mono=True)
        print(f"Resampled '{filename}' from {rate}Hz to {target_sr}Hz")
        # Save the resampled audio to a new file
        basename = os.path.basename(filename)
        output_filename = os.path.join(os.path.dirname(filename), "resampled_" + basename)
        output_filename = "resampled_" + filename
        sf.write(output_filename, audio, target_sr)
        print(f"Resampled audio saved as '{output_filename}'")
        return audio, rate, output_filename
    else:
        audio, rate = librosa.load(filename, sr=rate, mono=True)
        print(f"No resampling needed, loaded audio at {rate}Hz.")
        return audio, rate, filename

audio_data, sr, used_filename = load_audio("test_dirty.wav")

# Transcribe the audio using Whisper

def transcribe_audio(filename, audio):

    # Load the Whisper model
    model = whisper.load_model("tiny")
    # Transcribe the audio
    result = model.transcribe(audio, language="en")
    # Print the transcription
    print(f"Transcribed message: {result['text']}")
    # Save the transcription to a file
    saved_filename = filename.split(".")[0]
    with open(f"{saved_filename}.txt", "w") as f:
        f.write(result["text"])
    # Print the segments with timestamps
    for segment in result["segments"]:
        # Print the start and end times of each segment
        print(f"[{segment['start']:.2f} → {segment['end']:.2f}] {segment['text']}")

transcribe_audio(used_filename, audio_data)

# Acoustic analysis
import parselmouth
from parselmouth.praat import call

def analyse_audio(filename):
    # Load the audio file
    snd = parselmouth.Sound(filename)

    ## Vocal Energy
    # Extract intensity
    intensity = call(snd, "To Intensity", 75.0, 0.0)
    mean_intensity = call(intensity, "Get mean", 0, 0, "dB")
    min_intensity = call(intensity, "Get minimum", 0, 0, "Parabolic")
    max_intensity = call(intensity, "Get maximum", 0, 0, "Parabolic")
    
    if mean_intensity == 0:
        print("No intensity detected.")
    else:
        print(f"Mean Intensity: {mean_intensity:.2f} dB")
    print(f"Min Intensity: {min_intensity:.2f} dB")
    print(f"Max Intensity: {max_intensity:.2f} dB")
    
    if mean_intensity < 60:
        print("Low vocal energy detected.")
    elif mean_intensity >= 60 and mean_intensity < 70:
        print("Normal vocal energy detected.")
    elif mean_intensity >= 70 and mean_intensity < 85:
        print("High vocal energy detected.")
    else:
        print("Very high vocal energy detected.")

    # Plot intensity
    plt.figure(figsize=(10, 4))
    plt.plot(intensity.xs(), intensity.values.T, label="Intensity (dB)")
    plt.title("Intensity over Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Intensity (dB)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.clf()

    # Extract pitch
    pitch = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
    mean_pitch = call(pitch, "Get mean", 0, 0, "Hertz")
    min_pitch = call(pitch, "Get minimum", 0, 0, "Hertz", "Parabolic")
    max_pitch = call(pitch, "Get maximum", 0, 0, "Hertz", "Parabolic")
    pitch_range = max_pitch - min_pitch
    std_pitch = call(pitch, "Get standard deviation", 0, 0, "Hertz")
    
    print(f"Min Pitch: {min_pitch:.2f} Hz")
    print(f"Max Pitch: {max_pitch:.2f} Hz")
    print(f"Pitch Range: {pitch_range:.2f} Hz")
    if mean_pitch == 0:
        print("No pitch detected.")
    else:
        print(f"Mean Pitch: {mean_pitch:.2f} Hz")
    print(f"Pitch Standard Deviation: {std_pitch:.2f} Hz")

    # Plot pitch
    pitch_values = pitch.selected_array['frequency']
    pitch_values[pitch_values == 0] = np.nan  # Replace 0 with NaN for plotting
    pitch_times = pitch.xs()

    plt.figure(figsize=(10, 4))
    plt.plot(pitch_times, pitch_values, label="Pitch (Hz)")
    plt.xticks(ticks=np.arange(0, snd.duration, 0.5))
    plt.title("Pitch over Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Pitch (Hz)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.clf()

    ## Voice stability
    point_process = call(snd, "To PointProcess (periodic, cc)", 75, 500)

    # Extract jitter
    jitter = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)

    print(f"Jitter: {jitter:.4f}")

    if jitter < 0.005:
        print("Minimal jitter detected.")
    elif jitter >= 0.005 and jitter < 0.01:
        print("Normal jitter detected.")
    elif jitter >= 0.01 and jitter < 0.02:
        print("Moderate jitter detected.")
    else:
        print("High jitter detected.")

    # Extract shimmer
    shimmer = call([snd, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    print(f"Shimmer: {shimmer:.4f}")
    if shimmer < 0.035:
        print("Minimal shimmer detected. Normal vocal control.")
    elif shimmer >= 0.035 and shimmer < 0.045:
        print("Moderate shimmer detected.")
    else:
        print("High shimmer detected.")

analyse_audio("test_dirty.wav")

### Phoneme Analysis

role of this in the project. describe process of using MFA in terminal with an architecture à la up the top. 

The audio file and subsequent transcription need to feed into a dedicated program outside of the code, called Montreal Forced Alignment. The program provides timed occurrences of utterances at the word and phoneme level. It allows a deeper understanding of pronunciation and the components in speech in relation to time. The output is a text grid file, which is brought back into the code and analysed further. 

In [8]:
from textgrid import TextGrid

# Load the TextGrid file
def load_textgrid(filename):
    tg = TextGrid.fromFile(filename)

    # Inspect all tiers
    for tier in tg.tiers:
        print(f"Tier name: {tier.name}")
        for interval in tier.intervals:
            print(f"{interval.mark} - from {interval.minTime:.2f} to {interval.maxTime:.2f}")

load_textgrid("/Users/Jimdymock/Documents/Coding/Personal Project/Speech Analyser/MFA_Output/resampled_test_dirty.TextGrid")

## Speech fluency analysis
def detect_hesitation(filename, threshold=0.7):
    tg = TextGrid.fromFile(filename)
    fillers = {"uh", "um", "ah", "like", "you know", "i mean"}

    # Detecting long pauses near filler words
    for tier in tg.tiers:
        if "word" in tier.name.lower():
            print(f"Scanning tier: {tier.name}")

        intervals = tier.intervals

        for i, interval in enumerate(intervals):
            word = interval.mark.strip().lower()
            duration = interval.maxTime - interval.minTime

            if word in fillers:
                print(f"Filler detected: '{word}' - from {interval.minTime:.2f} to {interval.maxTime:.2f}")

            if word == '' and duration > threshold:
                prev_word = intervals[i - 1].mark.strip().lower() if i > 0 else ''
                next_word = intervals[i + 1].mark.strip().lower() if i < len(intervals) - 1 else ''
                    
                if prev_word in fillers or next_word in fillers:
                    print(f"Long pause detected between '{prev_word}' and '{next_word}'- from {interval.minTime:.2f} to {interval.maxTime:.2f}")

detect_hesitation("/Users/Jimdymock/Documents/Coding/Personal Project/Speech Analyser/MFA_Output/resampled_test_dirty.TextGrid")

Tier name: words
 - from 0.00 to 0.75
hello - from 0.75 to 1.36
 - from 1.36 to 2.78
this - from 2.78 to 3.06
is - from 3.06 to 3.41
like - from 3.41 to 3.81
 - from 3.81 to 3.85
an - from 3.85 to 3.96
example - from 3.96 to 4.48
 - from 4.48 to 4.66
of - from 4.66 to 4.97
 - from 4.97 to 6.04
bad - from 6.04 to 6.43
speech - from 6.43 to 6.97
 - from 6.97 to 7.95
Tier name: phones
 - from 0.00 to 0.75
h - from 0.75 to 0.84
ə - from 0.84 to 0.89
l - from 0.89 to 1.00
əw - from 1.00 to 1.36
 - from 1.36 to 2.78
d̪ - from 2.78 to 2.86
ɪ - from 2.86 to 2.93
s - from 2.93 to 3.06
i - from 3.06 to 3.28
z - from 3.28 to 3.41
l - from 3.41 to 3.44
aj - from 3.44 to 3.59
k - from 3.59 to 3.81
 - from 3.81 to 3.85
a - from 3.85 to 3.93
n - from 3.93 to 3.96
ə - from 3.96 to 4.00
ɡ - from 4.00 to 4.05
z - from 4.05 to 4.13
æ - from 4.13 to 4.24
m - from 4.24 to 4.30
p - from 4.30 to 4.41
ə - from 4.41 to 4.45
ɫ - from 4.45 to 4.48
 - from 4.48 to 4.66
ɒ - from 4.66 to 4.94
v - from 4.94 to 4.97


#### * * Final task to clean file - work out the best way for files to be handled: original audio file, the text file that will be written, and then the access to the textgrid file to do phonemic analysis